# Visualize Denoising Metric Comparisons

This notebook collects metric outputs from `results_smooth`, `results_magic`, and `results_log_norm`, then compares them side by side.

It is designed to work directly from the folder structure in `testing_different_denoisings` and should keep working if you add more samples later.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')


In [ ]:
BASE_DIR = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Breast_cancer_Xenium5K/01_InSituCNV/testing_different_denoisings')

RESULT_DIRS = {
    'smooth': BASE_DIR / 'results_smooth',
    'magic': BASE_DIR / 'results_magic',
    'log_norm': BASE_DIR / 'results_log_norm',
}

sample_key = '1105BL'
method_order = ['smooth', 'magic', 'log_norm']

BASE_DIR

In [ ]:
def sample_prefix(path: Path) -> str:
    return path.name.split('__', 1)[0]


def load_result_tables(base_dir: Path, result_dirs: dict[str, Path]) -> tuple[pd.DataFrame, dict[str, dict[str, dict[str, Path]]]]:
    file_registry: dict[str, dict[str, dict[str, Path]]] = {}
    metrics_frames = []

    for method, folder in result_dirs.items():
        method_files: dict[str, dict[str, Path]] = {}
        for csv_path in sorted(folder.glob('*.csv')):
            prefix = sample_prefix(csv_path)
            suffix = csv_path.name.split('__', 1)[1]
            method_files.setdefault(prefix, {})[suffix] = csv_path

            if suffix == 'metrics_summary.csv':
                df = pd.read_csv(csv_path)
                df['method'] = method
                df['sample_key'] = prefix
                df['source_file'] = str(csv_path)
                metrics_frames.append(df)

        file_registry[method] = method_files

    metrics_df = pd.concat(metrics_frames, ignore_index=True)
    metrics_df['method'] = pd.Categorical(metrics_df['method'], categories=method_order, ordered=True)
    metrics_df = metrics_df.sort_values(['sample_key', 'method']).reset_index(drop=True)
    return metrics_df, file_registry


metrics_df, file_registry = load_result_tables(BASE_DIR, RESULT_DIRS)

available_samples = sorted(metrics_df['sample_key'].unique())
print('Available sample keys:', available_samples)

if sample_key not in available_samples:
    raise ValueError(f'sample_key={sample_key!r} not found. Choose one of: {available_samples}')

sample_metrics = metrics_df.loc[metrics_df['sample_key'] == sample_key].copy()
display(sample_metrics)

## Summary Comparison

The table below reshapes the metrics summary into a tidy format for plotting.

In [ ]:
metric_columns = [
    'auc_gain', 'auc_loss', 'auc_mean',
    'precision_gain', 'precision_loss', 'precision_mean',
    'recall_gain', 'recall_loss', 'recall_mean',
    'f1_gain', 'f1_loss', 'f1_mean',
]

tidy_metrics = sample_metrics.melt(
    id_vars=['sample_key', 'sample_name', 'method', 'wes_threshold', 'insitu_threshold'],
    value_vars=metric_columns,
    var_name='metric',
    value_name='value',
)
tidy_metrics[['metric_family', 'state']] = tidy_metrics['metric'].str.split('_', n=1, expand=True)

display(tidy_metrics.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
family_order = ['auc', 'precision', 'recall', 'f1']
state_order = ['gain', 'loss', 'mean']

for ax, family in zip(axes.flat, family_order):
    plot_df = tidy_metrics.loc[tidy_metrics['metric_family'] == family].copy()
    sns.barplot(
        data=plot_df,
        x='state',
        y='value',
        hue='method',
        order=state_order,
        hue_order=method_order,
        ax=ax,
    )
    ax.set_title(f'{family.upper()} metrics')
    ax.set_xlabel('')
    ax.set_ylabel('score')
    ax.set_ylim(0, 1.05)
    if ax is not axes.flat[0]:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

axes.flat[0].legend(title='method', loc='lower right')
plt.show()

In [ ]:
selected_thresholds = sample_metrics[
    ['method', 'wes_threshold', 'insitu_threshold', 'auc_mean', 'f1_mean', 'precision_mean', 'recall_mean']
].set_index('method')
display(selected_thresholds)

## Threshold Grid Curves

These plots show how each denoising method behaves across the stored threshold grids and highlight the selected threshold from `metrics_summary.csv`.

In [ ]:
def load_grid(method: str, suffix: str) -> pd.DataFrame:
    path = file_registry[method][sample_key][suffix]
    return pd.read_csv(path)


insitu_grids = []
wes_grids = []

for method in method_order:
    insitu_df = load_grid(method, 'insitu_threshold_f1_grid.csv')
    insitu_df['method'] = method
    insitu_grids.append(insitu_df)

    wes_df = load_grid(method, 'wes_threshold_auc_grid.csv')
    wes_df['method'] = method
    wes_grids.append(wes_df)

insitu_grid_df = pd.concat(insitu_grids, ignore_index=True)
wes_grid_df = pd.concat(wes_grids, ignore_index=True)

display(insitu_grid_df.head())
display(wes_grid_df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

sns.lineplot(
    data=insitu_grid_df,
    x='insitu_threshold',
    y='f1_mean',
    hue='method',
    hue_order=method_order,
    marker='o',
    ax=axes[0],
)
for method in method_order:
    row = selected_thresholds.loc[method]
    axes[0].axvline(row['insitu_threshold'], linestyle='--', alpha=0.4)
axes[0].set_title('In situ threshold vs F1 mean')
axes[0].set_ylabel('F1 mean')
axes[0].set_xlabel('in situ threshold')
axes[0].set_ylim(0, 1.05)

sns.lineplot(
    data=wes_grid_df,
    x='wes_threshold',
    y='auc_mean',
    hue='method',
    hue_order=method_order,
    marker='o',
    ax=axes[1],
)
for method in method_order:
    row = selected_thresholds.loc[method]
    axes[1].axvline(row['wes_threshold'], linestyle='--', alpha=0.4)
axes[1].set_title('WES threshold vs AUC mean')
axes[1].set_ylabel('AUC mean')
axes[1].set_xlabel('WES threshold')
axes[1].set_ylim(0, 1.05)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

sns.lineplot(
    data=insitu_grid_df,
    x='insitu_threshold',
    y='accuracy_3class',
    hue='method',
    hue_order=method_order,
    marker='o',
    ax=axes[0],
)
axes[0].set_title('In situ threshold vs 3-class accuracy')
axes[0].set_ylabel('accuracy')
axes[0].set_xlabel('in situ threshold')
axes[0].set_ylim(0, 1.05)

sns.lineplot(
    data=wes_grid_df,
    x='wes_threshold',
    y='n_truth_gain',
    hue='method',
    hue_order=method_order,
    marker='o',
    ax=axes[1],
)
sns.lineplot(
    data=wes_grid_df,
    x='wes_threshold',
    y='n_truth_loss',
    hue='method',
    hue_order=method_order,
    marker='s',
    linestyle='--',
    legend=False,
    ax=axes[1],
)
axes[1].set_title('Truth events retained across WES thresholds')
axes[1].set_ylabel('count')
axes[1].set_xlabel('WES threshold')

plt.show()

## Confusion Matrices

Counts and row-normalized percentages are shown for each method to make class-specific behavior easier to compare.

In [ ]:
def load_confusion(method: str, suffix: str) -> pd.DataFrame:
    path = file_registry[method][sample_key][suffix]
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: 'truth'})
    return df.set_index('truth')


count_matrices = {method: load_confusion(method, 'confusion_matrix_counts.csv') for method in method_order}
pct_matrices = {method: load_confusion(method, 'confusion_matrix_row_pct.csv') for method in method_order}

count_matrices['smooth']

In [ ]:
fig, axes = plt.subplots(2, len(method_order), figsize=(18, 10), constrained_layout=True)

for idx, method in enumerate(method_order):
    sns.heatmap(count_matrices[method], annot=True, fmt='.0f', cmap='Blues', cbar=idx == len(method_order) - 1, ax=axes[0, idx])
    axes[0, idx].set_title(f'{method}: counts')
    axes[0, idx].set_xlabel('predicted')
    axes[0, idx].set_ylabel('truth')

    sns.heatmap(pct_matrices[method], annot=True, fmt='.1f', cmap='YlGnBu', cbar=idx == len(method_order) - 1, ax=axes[1, idx])
    axes[1, idx].set_title(f'{method}: row %')
    axes[1, idx].set_xlabel('predicted')
    axes[1, idx].set_ylabel('truth')

plt.show()

## Optional: Inspect Per-bin Comparison Rows

Use this section if you want to compare the bin-level `wes_segmean` and `insitu_value` tables exported for each method.

In [ ]:
comparison_frames = []

for method in method_order:
    path = file_registry[method][sample_key]['comparison_rows.csv']
    df = pd.read_csv(path)
    df['method'] = method
    comparison_frames.append(df)

comparison_df = pd.concat(comparison_frames, ignore_index=True)
display(comparison_df.head())

In [ ]:
fig, axes = plt.subplots(1, len(method_order), figsize=(20, 5), sharex=True, sharey=True, constrained_layout=True)

for ax, method in zip(axes, method_order):
    plot_df = comparison_df.loc[comparison_df['method'] == method]
    sns.scatterplot(data=plot_df, x='wes_segmean', y='insitu_value', hue='truth', style='pred', palette='deep', ax=ax)
    ax.set_title(method)
    ax.set_xlabel('WES segmean')
    ax.set_ylabel('in situ value')

handles, labels = axes[-1].get_legend_handles_labels()
for ax in axes[:-1]:
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
axes[-1].legend(handles, labels, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.show()